In [1]:
import torch
print(f"Is CUDA available? {torch.cuda.is_available()}")
print(f"Device Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

Is CUDA available? True
Device Name: NVIDIA GeForce RTX 3050 Laptop GPU


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from PIL import Image
import json
import os

# 1. Hardware Force: Only float16 on CUDA (Best for RTX 3050 4GB)
device = "cuda"
dtype = torch.float16 # Using the new 'dtype' parameter

print(f"🚀 Initializing Moondream 2 on: {torch.cuda.get_device_name(0)}")

model_id = "vikhyatk/moondream2"
revision = "2025-01-09" 

# 2. Load Model & Tokenizer using 'dtype'
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    trust_remote_code=True, 
    revision=revision,
    dtype=dtype,           # UPDATED: Replaced torch_dtype with dtype
    device_map={"": device}
)
tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision)

# 3. Setup Image
image_path = r"D:\Y4 Research\datasets\dietary Images\Set1\542.png"
image = Image.open(image_path)
if image.mode != "RGB":
    image = image.convert("RGB")
width, height = image.size

# 4. Extract Claims using your specific prompt
print("📝 Extracting health claims...")
image_embeds = model.encode_image(image)

# YOUR SPECIFIC PROMPT:
prompt = "List all health claims on this label. Return as a simple comma-separated list."

answer = model.answer_question(image_embeds, prompt, tokenizer)
# Clean the list (handles potential extra spaces or trailing commas)
claims_list = [c.strip() for c in answer.split(',') if c.strip()]

# 5. Detect Bounding Boxes for each extracted claim
structured_results = []
for claim in claims_list:
    print(f"🔍 Locating: {claim}")
    # Moondream 2 detect returns normalized coordinates [0-1000]
    result = model.detect(image, claim)
    
    # MD2 usually returns {'objects': [...]} or a list directly depending on version
    objects = result.get("objects", []) if isinstance(result, dict) else result
    
    for obj in objects:
        # Scale to image pixels
        x_min = (obj["x_min"] / 1000) * width
        y_min = (obj["y_min"] / 1000) * height
        x_max = (obj["x_max"] / 1000) * width
        y_max = (obj["y_max"] / 1000) * height
        
        structured_results.append({
            "claim": claim,
            "coordinates_pixel": {
                "ymin": int(y_min), "xmin": int(x_min), 
                "ymax": int(y_max), "xmax": int(x_max)
            }
        })

# 6. Save JSON Output
output_dir = r"E:\sample_jsons"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "542_claims.json")

with open(output_path, "w") as f:
    json.dump(structured_results, f, indent=4)

print(f"✅ Success! Results saved to: {output_path}")
print(f"Summary of claims found: {claims_list}")

🚀 Initializing Moondream 2 on: NVIDIA GeForce RTX 3050 Laptop GPU


model.safetensors:   0%|          | 0.00/3.85G [00:00<?, ?B/s]